# Download Protein Structures for each protein

In [ ]:
import requests

In [ ]:
import requests
import pandas as pd
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import os

def fetch_best_structure(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/search?query={uniprot_id}&fields=structure_3d&format=json"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception:
        return {
            'accession': uniprot_id,
            'best_pdb_id': '',
            'resolution': '',
            'full_metadata': ''
        }

    best_entry = None
    best_resolution = float("inf")

    try:
        structures = data['results'][0].get('uniProtKBCrossReferences', [])
        for entry in structures:
            if entry.get('database') == 'PDB':
                for prop in entry.get('properties', []):
                    if prop.get('key', '').lower() == 'resolution':
                        try:
                            resolution = float(prop['value'].strip(' A'))
                            if resolution < best_resolution:
                                best_resolution = resolution
                                best_entry = entry
                        except ValueError:
                            continue
    except (IndexError, KeyError):
        pass

    return {
        'accession': uniprot_id,
        'best_pdb_id': best_entry['id'] if best_entry else '',
        'resolution': best_resolution if best_entry else '',
        'full_metadata': json.dumps(best_entry) if best_entry else ''
    }

# 🔄 BATCH-SAVING VERSION
def run_concurrent_batchsave(uniprot_ids, out_csv="best_structures.csv", max_workers=32, batch_size=1000):
    results = []
    counter = 0
    batch_id = 0

    # Remove existing file if restarting
    if os.path.exists(out_csv):
        os.remove(out_csv)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_uid = {executor.submit(fetch_best_structure, uid): uid for uid in uniprot_ids}

        for future in tqdm(as_completed(future_to_uid), total=len(uniprot_ids), desc="Fetching UniProt structures"):
            try:
                results.append(future.result())
            except Exception:
                results.append({
                    'accession': future_to_uid[future],
                    'best_pdb_id': '',
                    'resolution': '',
                    'full_metadata': ''
                })

            counter += 1

            # Save every batch_size entries
            if counter % batch_size == 0:
                df = pd.DataFrame(results)
                df.to_csv(out_csv, mode='a', index=False, header=not os.path.exists(out_csv))
                results = []  # Clear the buffer
                batch_id += 1

    # Final write for any remaining results
    if results:
        df = pd.DataFrame(results)
        df.to_csv(out_csv, mode='a', index=False, header=not os.path.exists(out_csv))

    print(f"Done. Saved in batches of {batch_size} to: {out_csv}")

In [ ]:
# --- Example Usage ---

# Replace this with your 250k+ UniProt IDs
accessions = list(set(list(merged['protein_id'].unique()) + list(metadata_test['protein_id'].unique())))  # or load from file

run_concurrent_batchsave(accessions, out_csv="best_uniprot_structures.csv")

Fetching UniProt structures: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201777/201777 [41:10<00:00, 81.67it/s]


Done. Saved in batches of 1000 to: best_uniprot_structures.csv


AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
import pandas as pd

# Load your CSV
structures = pd.read_csv('best_uniprot_structures.csv')

# --- Save UniProt IDs with no PDB ID ---
no_pdb = structures[structures['best_pdb_id'].isna() | (structures['best_pdb_id'] == '')]
no_pdb_ids = no_pdb['accession'].tolist()

with open("uniprot_no_pdb.txt", "w") as f:
    for acc in no_pdb_ids:
        f.write(f"{acc}\n")

# --- Save valid PDB IDs ---
has_pdb = structures[structures['best_pdb_id'].notna() & (structures['best_pdb_id'] != '')]
pdb_ids = has_pdb['best_pdb_id'].tolist()

with open("uniprot_pdb_id.txt", "w") as f:
    for pdb in pdb_ids:
        f.write(f"{pdb}\n")

### Download PDB files

In [ ]:
wget https://www.rcsb.org/scripts/batch_download.sh

In [ ]:
mkdir pdb_files

In [ ]:
./batch_download.sh -p -f uniprot_pdb_id.txt -o pdb_files/ 

In [ ]:
import os
from os.path import basename
import biotite.database.rcsb as rcsb
from tqdm import tqdm

# ✅ Correct file reading
with open("uniprot_pdb_id.txt", "r") as f:
    pdb_ids = [line.strip() for line in f if line.strip()]

def fast_download_pdbs(pdb_ids, output_dir="pdb_files"):
    os.makedirs(output_dir, exist_ok=True)

    # Filter out already downloaded files
    to_download = []
    for pdb_id in pdb_ids:
        file_path = os.path.join(output_dir, f"{pdb_id.lower()}.pdb")
        if not os.path.exists(file_path):
            to_download.append(pdb_id)

    print(f"Downloading {len(to_download)} new PDBs...")

    # Download in batches
    batch_size = 50
    for i in tqdm(range(0, len(to_download), batch_size), desc="Fetching PDBs"):
        batch = to_download[i:i + batch_size]
        try:
            rcsb.fetch(batch, format="pdb", target_path=output_dir)
        except Exception as e:
            print(f"❌ Failed batch: {batch}\n   Error: {e}")

    print("✅ Done!")

# ✅ Run the downloader
fast_download_pdbs(pdb_ids)

In [ ]:
grep -l "Archive" pdb_files/* > uniprot_no_pdb_file.txt

In [ ]:
sed -i 's|pdb_files/||g' uniprot_no_pdb_file.txt
sed -i 's|.pdb||g' uniprot_no_pdb_file.txt

In [ ]:
import pandas as pd

pdb_ids = pd.read_csv('best_uniprot_structures.csv')

In [ ]:
with open('uniprot_no_pdb_file.txt', 'r') as f:
    bad_pdbs = [line.strip() for line in f if line.strip()]

In [ ]:
len(bad_pdbs)

572

In [ ]:
pdb_ids

,accession,best_pdb_id,resolution,full_metadata
0,Q9VMP9,NaN,NaN,NaN
1,P76553,NaN,NaN,NaN
2,Q8GS71,NaN,NaN,NaN
3,O13747,NaN,NaN,NaN
4,P47864,NaN,NaN,NaN
...,...,...,...,...
201772,P54722,NaN,NaN,NaN
201773,Q9BHL0,NaN,NaN,NaN
201774,A8E653,NaN,NaN,NaN
201775,Q4QNA8,NaN,NaN,NaN


In [ ]:
matching_accessions = pdb_ids[pdb_ids['best_pdb_id'].isin(bad_pdbs)]['accession'].tolist()

In [ ]:
len(matching_accessions)

2604

### Downloading AlphaFoldDB

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [ ]:
with open('uniprot_no_pdb.txt', 'r') as f:
    af_uniprots = [line.strip() for line in f if line.strip()]

In [ ]:
len(af_uniprots)

176344

In [ ]:
len(af_uniprots + matching_accessions)

178948

In [ ]:
with open("alphafold_cif_manifest.txt", "w") as f:
    for uid in (af_uniprots + matching_accessions):
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{uid}-F1-model_v4.cif\n")

In [ ]:
cat alphafold_cif_manifest.txt | gsutil -m cp -I af_db/

In [ ]:
import os

# Combined list of all expected accessions
all_expected = set(af_uniprots + matching_accessions)

# Get all filenames in af_db/ and extract accessions
downloaded_files = os.listdir("af_db")
downloaded_accessions = set(
    filename.split("-")[1]  # Extract accession from 'AF-<ACCESSION>-F1-model_v4.cif'
    for filename in downloaded_files
    if filename.endswith(".cif")
)

# Find accessions that are missing
missing_accessions = sorted(all_expected - downloaded_accessions)

print(f"Expected: {len(all_expected)}")
print(f"Downloaded: {len(downloaded_accessions)}")
print(f"Missing: {len(missing_accessions)}")

# Save to file
with open("missing_af_accessions.txt", "w") as f:
    for acc in missing_accessions:
        f.write(f"{acc}\n")

Expected: 178948
Downloaded: 171279
Missing: 7669


In [ ]:
find af_db/ -type f -name "*.cif" | xargs -n 1000 pigz --best
find pdb_files/ -type f -name "*.pdb" | xargs -n 1000 pigz --best

In [ ]:
cd ../../
git lfs install
git clone https://huggingface.co/datasets/wanglab/cafa5

mkdir cafa5/structures
cp -r af_db/ ../../cafa5/structures/
cp -r pdb_files/ ../../cafa5/structures/

cd ../../cafa5/

git add structures/
git commit -m "Add PDB and AlphaFold structures"
git push

In [ ]:
mkdir -p af_shards

i=0
shard=0

for file in af_db/*.cif.gz; do
    # Create a new folder every 5000 files
    if (( i % 5000 == 0 )); then
        shard_dir="af_shards/shard_${shard}"
        mkdir -p "$shard_dir"
        ((shard++))
    fi

    cp "$file" "$shard_dir/"
    ((i++))
done

In [ ]:
cd af_shards

for dir in shard_*; do
    tar -czf "${dir}.tar.gz" "$dir"
    rm -r "$dir"  # optional cleanup
done

cd ..

# Updating CAFA5 Dataset to reflect those without structures 

In [ ]:
with open('missing_af_accessions.txt', 'r') as f:
    missing_accessions = [line.strip() for line in f if line.strip()]

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

In [ ]:
cafa5_train = ds['train'].to_pandas()

In [ ]:
missing_df = cafa5_train[cafa5_train['protein_id'].isin(missing_accessions)]

In [ ]:
missing_df

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
7,A0A0B4K7J2,E3 SUMO-protein ligase RanBP2 (EC 2.3.2.-) (35...,E3 SUMO-protein ligase (By similarity). Compon...,Drosophila melanogaster (Fruit fly),2718.0,"Nucleus, nuclear pore complex {ECO:0000250|Uni...",MFTTRKEVDAHVHKMLGKLQPGRERDIKGLAVARLYMKVQEYPKAI...,"[GO:0006606, GO:0008152, GO:0032241, GO:001995...","[GO:0008150, GO:0008152, GO:0051179, GO:004851...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0016020, GO:001250..."
86,A0A0B4K6M2,"Rab3 interacting molecule, isoform F (Rab3 int...",None,Drosophila melanogaster (Fruit fly),2798.0,Synapse {ECO:0000256|ARBA:ARBA00034103}.,MDEMPDLSHLTPHERMQIENVLMRQKQEEEKQNEIMRRKQDEVVTL...,"[GO:0007267, GO:0048869, GO:0065008, GO:004885...","[GO:0008150, GO:0065007, GO:0050789, GO:003250...",None,"[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
87,A0A0B4K6S2,"Down syndrome cell adhesion molecule 1, isofor...",None,Drosophila melanogaster (Fruit fly),2036.0,None,MNMPNERLKWLMLFAAVALIACGSQTLAANPPDADQKGPVFLKEPT...,"[GO:0009581, GO:0032989, GO:0009605, GO:004001...","[GO:0008150, GO:0040011, GO:0051179, GO:005078...","[GO:0003674, GO:0060089, GO:0005488, GO:000382...","[GO:0005575, GO:0110165, GO:0044297, GO:003647..."
90,A0A0B4K6Y9,"Down syndrome cell adhesion molecule 1, isofor...",None,Drosophila melanogaster (Fruit fly),2020.0,None,MNMPNERLKWLMLFAAVALIACGSQTLAANPPDADQKGPVFLKEPT...,"[GO:0009581, GO:0032989, GO:0009605, GO:004001...","[GO:0008150, GO:0040011, GO:0051179, GO:005078...","[GO:0003674, GO:0060089, GO:0005488, GO:000382...","[GO:0005575, GO:0110165, GO:0044297, GO:003647..."
91,A0A0B4K6Z3,"Down syndrome cell adhesion molecule 1, isofor...",None,Drosophila melanogaster (Fruit fly),2019.0,None,MNMPNERLKWLMLFAAVALIACGSQTLAANPPDADQKGPVFLKEPT...,"[GO:0009581, GO:0032989, GO:0009605, GO:004001...","[GO:0008150, GO:0040011, GO:0051179, GO:005078...","[GO:0003674, GO:0060089, GO:0005488, GO:000382...","[GO:0005575, GO:0110165, GO:0044297, GO:003647..."
...,...,...,...,...,...,...,...,...,...,...,...
133442,U5TQC3,UL14,None,Human herpesvirus 1 (HHV-1) (Human herpes simp...,219.0,Host cytoplasm {ECO:0000256|ARBA:ARBA00004192}...,MDRDAAHAALRRRLAETHLRAEIYKDQTLQLHREGVSTQDPRFVGA...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None
133477,U5TMQ7,DNA packaging protein UL33,None,Human herpesvirus 1 (HHV-1) (Human herpes simp...,130.0,None,MAGREGRTRQRTLRDTIPDCALRSQTLESLDARYVSRDGAHNAAVW...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None
133478,V6BMM3,Proteolysis tag peptide encoded by tmRNA Caulo...,None,Caulobacter vibrioides (strain ATCC 19089 / CI...,13.0,None,ANDNFAEEFAVAA,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None
133487,R4JMG7,Peptidase M15A,None,Burkholderia phage ST79,149.0,None,MQLTDHFTLEELTASDVARTRQIDNTPSAATVENLRRLAQTLEQAR...,"[GO:0003674, GO:0140096, GO:0016787, GO:000382...",None,"[GO:0003674, GO:0003824, GO:0140096, GO:001678...",None


In [ ]:
cafa5_test = ds['test'].to_pandas()
missing_df_test = cafa5_test[cafa5_test['protein_id'].isin(missing_accessions)]
missing_df_test

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
276,Q5SSE9,ATP-binding cassette sub-family A member 13 (E...,May mediate the cholesterol and gangliosides t...,Mus musculus (Mouse),5034.0,Cytoplasmic vesicle membrane {ECO:0000269|PubM...,MGHAGRQFQALLWKNWICRLRHPVLSLAEFFWPCILFMILTVLRFQ...,[None],[None],[None],[None]
337,Q8VHN7,Adhesion G-protein coupled receptor V1 (ADGRV1...,G-protein coupled receptor which has an essent...,Mus musculus (Mouse),6298.0,Cell membrane {ECO:0000269|PubMed:15606908}; M...,MSVTSEPGMISSFLLVYLSTLFISFVFGEAEIRFTGQTEFFVNETS...,[None],[None],[None],[None]
417,B7ZCC9,Adhesion G-protein coupled receptor G4 (G-prot...,Orphan receptor.,Mus musculus (Mouse),3073.0,Membrane {ECO:0000255}; Multi-pass membrane pr...,MRKHILHQRLCGLILVSSFIFLTDSLSLKGKRLDFYGEGKAYVSLT...,[None],[None],[None],[None]
498,P83855,Putative sperm adenylate cyclase (EC 4.6.1.1),None,Mus musculus (Mouse),21.0,None,GVYMEIGRCRXEAXRRRKEAV,[None],[None],[None],[None]
551,E9Q394,A-kinase anchor protein 13 (AKAP-13) (AKAP-Lbc),Scaffold protein that plays an important role ...,Mus musculus (Mouse),2776.0,"Cytoplasm, cytosol {ECO:0000250|UniProtKB:Q128...",MKLSPQQAPLYGDCVVTVLLAEEDKVEDDAIFYLIFSGSTLYHCTS...,[None],[None],[None],[None]
...,...,...,...,...,...,...,...,...,...,...,...
141449,J9VQH1,ATP-dependent permease YOR1 (EC 7.2.2.-) (ABC ...,Transmembrane transporter (By similarity). May...,Cryptococcus neoformans var. grubii serotype A...,1512.0,Extracellular vesicle membrane {ECO:0000269|Pu...,MSPLLPTHWGASAPQNEPTLPSPSHSVSTRVGDEEKLRRSEGSDGE...,[None],[None],[None],[None]
141509,P39088,Hemolytic toxin (Cytolysin) (DELTA-stichotoxin),Pore-forming protein that forms cations-select...,Heteractis magnifica (Magnificent sea anemone)...,54.0,Secreted {ECO:0000250|UniProtKB:B9W5G6}. Nemat...,SAALAGTIIAGASLGFQILDKVLGELGKVSRKIAIGVDNESIGSNT...,[None],[None],[None],[None]
141746,Q1LYH4,merged,None,None,NaN,None,None,[None],[None],[None],[None]
141759,Q9JI18,None,None,mus musculus[All Names],NaN,None,MSQLLLAILTLSGLLPNAEVLIVGANQDQHLCDPGEFLCHDHVTCV...,[None],[None],[None],[None]


In [ ]:
len(set(list(missing_df['protein_id']) + list(missing_df_test['protein_id'])))

7669

In [ ]:
structures = pd.read_csv('best_uniprot_structures.csv')
structures = structures.rename(columns={"accession": "protein_id"})

In [ ]:
cafa5_train_struc = cafa5_train.merge(structures, on='protein_id', how="left")
cafa5_test_struc = cafa5_test.merge(structures, on='protein_id', how="left")

In [ ]:
import os
import numpy as np

def get_structure_path(row):
    pdb_id = row['best_pdb_id']
    protein_id = row['protein_id']
    
    # 1. Check PDB structure
    if pd.notna(pdb_id):
        pdb_path = f"pdb_files/{pdb_id}.pdb.gz"
        if os.path.exists(pdb_path):
            return pdb_path

    # 2. Check AlphaFold structure
    af_path = f"af_db/AF-{protein_id}-F1-model_v4.cif.gz"
    if os.path.exists(af_path):
        return af_path

    # 3. Nothing found
    return np.NaN

# Apply function to DataFrame
cafa5_train_struc['structure_path'] = cafa5_train_struc.apply(get_structure_path, axis=1)
cafa5_test_struc['structure_path'] = cafa5_test_struc.apply(get_structure_path, axis=1)

In [ ]:
len(cafa5_train_struc[cafa5_train_struc['structure_path'].isna()]), len(cafa5_test_struc[cafa5_test_struc['structure_path'].isna()])

(6943, 1173)

In [ ]:
cafa5_train_struc = cafa5_train_struc.drop(columns=['best_pdb_id', 'resolution', 'full_metadata'])
cafa5_test_struc = cafa5_test_struc.drop(columns=['best_pdb_id', 'resolution', 'full_metadata'])

In [ ]:
cafa5_train_struc

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0003824, GO:0016491, GO:001670...","[GO:0005575, GO:0110165, GO:0005622, GO:004322...","[IPR001128, IPR017972, IPR002401, IPR008069, I...",af_db/AF-A0A087X1C5-F1-model_v4.cif.gz
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0008150, GO:0050789, GO:0065007, GO:004858...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",NaN,af_db/AF-A0A0B4J2F0-F1-model_v4.cif.gz
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,"Together with MOM1 and PIAL2, regulates transc...",Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008150, GO:0008152, GO:0065007, GO:004851...","[GO:0003674, GO:0003824, GO:0016740, GO:014009...",None,"[IPR004181, IPR013083]",af_db/AF-A0A0A7EPL0-F1-model_v4.cif.gz
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0008150, GO:0002376, GO:0009987, GO:006500...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0071944, GO:001602...","[IPR007110, IPR036179, IPR013783, IPR050488, I...",af_db/AF-A0A0B4J1G0-F1-model_v4.cif.gz
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0008150, GO:0040011, GO:0002376, GO:000998...","[GO:0003674, GO:0098772, GO:0005488, GO:014067...",None,[IPR031713],af_db/AF-A0A0B4J1N3-F1-model_v4.cif.gz
...,...,...,...,...,...,...,...,...,...,...,...,...,...
133491,V9XTM1,Erythrocyte-binding antigen 175,None,Plasmodium sp. chimpanzee clade C3,631.0,None,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR042202, IPR008602]",af_db/AF-V9XTM1-F1-model_v4.cif.gz
133492,W5IDC3,Autonomous cohesin,None,Ruminococcus flavefaciens,201.0,None,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,NaN,pdb_files/4WKZ.pdb.gz
133493,Q9YWN5,VP9,None,Banna virus (BAV),283.0,None,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK...,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515, GO:0042802]",None,"[IPR015072, IPR043116]",pdb_files/1W9Z.pdb.gz
133494,Q9Z7K1,Flagellar motor switch Domain/YscQ family (Typ...,None,Chlamydia pneumoniae (Chlamydophila pneumoniae),371.0,None,MAVAADSSASWLKSRNNFLSSLGKTEEQVAAPEFPKELCQHKIREK...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR001543, IPR036429, IPR013385]",af_db/AF-Q9Z7K1-F1-model_v4.cif.gz


# Getting InterPro Domains for the Proteins

In [ ]:
from datasets import load_dataset
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from tqdm import tqdm

def fetch_one_protein(accession, max_retries=3):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/InterPro/protein/UniProt/{accession}"
    headers = {"Accept": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 204:
                return []
            r.raise_for_status()

            results = r.json().get("results", [])
            rows = []

            for item in results:
                ipr_id = item.get("metadata", {}).get("accession")
                entry_name = item.get("metadata", {}).get("name")
                ipr_type = item.get("metadata", {}).get("type")

                all_fragments = []
                for loc in item.get("proteins", []):  # fixed: must go through item['proteins']
                    for entry in loc.get("entry_protein_locations", []):
                        all_fragments.extend(entry.get("fragments", []))

                if all_fragments:
                    start = min(f["start"] for f in all_fragments if "start" in f)
                    end = max(f["end"] for f in all_fragments if "end" in f)

                    rows.append({
                        "accession": accession,
                        "interpro_id": ipr_id,
                        "entry_name": entry_name,
                        "type": ipr_type,
                        "start": start,
                        "end": end,
                        "n_fragments": len(all_fragments)
                    })
            return rows

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Failed for {accession}: {e}")
            sleep(2 * (attempt + 1))
    return []

def fetch_all_interpro_parallel(
    all_accessions,
    output_file="interpro_domains.tsv",
    interpro_lookup_file="interpro_lookup.tsv",
    max_threads=32
):
    protein_rows = []
    interpro_lookup = {}

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(fetch_one_protein, acc): acc for acc in all_accessions}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching InterPro"):
            rows = future.result()
            protein_rows.extend(rows)

    # Extract lookup table
    for row in protein_rows:
        ipr = row["interpro_id"]
        if ipr not in interpro_lookup:
            interpro_lookup[ipr] = {
                "entry_name": row["entry_name"],
                "type": row["type"]
            }

    # Save data
    pd.DataFrame(protein_rows).to_csv(output_file, sep="\t", index=False)
    pd.DataFrame([
        {"interpro_id": ipr, "entry_name": val["entry_name"], "type": val["type"]}
        for ipr, val in interpro_lookup.items()
    ]).to_csv(interpro_lookup_file, sep="\t", index=False)

    print(f"✅ Done. Fetched {len(protein_rows)} entries across {len(all_accessions)} proteins")

# ---------- Main execution ----------
if __name__ == "__main__":
    ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
    cafa5_train = ds["train"].to_pandas()
    cafa5_test = ds["test"].to_pandas()
    cafa5_test_super = ds["test_superset"].to_pandas()

    proteins = set(list(cafa5_train["protein_id"]) + list(cafa5_test["protein_id"]) + list(cafa5_test_super["protein_id"]))

    print(f"Fetched {len(proteins)} proteins")

    fetch_all_interpro_parallel(
        list(proteins),
        output_file="interpro_domains.tsv",
        interpro_lookup_file="interpro_lookup.tsv",
        max_threads=32
    )

In [ ]:
from datasets import load_dataset
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

In [ ]:
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

In [ ]:
import pandas as pd

In [ ]:
interpro = pd.read_csv('interpro_domains.tsv', sep='\t')

In [ ]:
interpro

,accession,interpro_id,entry_name,type,start,end,n_fragments
0,A0A8I5Y4S4,IPR000697,WH1/EVH1 domain,domain,1,111,1
1,A0A8I5Y4S4,IPR011993,PH-like domain superfamily,homologous_superfamily,1,117,1
2,A0A8I5Y4S4,IPR014885,VASP tetramerisation,domain,767,801,1
3,A0A8I5Y4S4,IPR038023,VASP tetramerisation domain superfamily,homologous_superfamily,765,803,1
4,Q67UU0,IPR003607,HD/PDEase domain,domain,229,359,1
...,...,...,...,...,...,...,...
849616,P93285,IPR011759,"Cytochrome C oxidase subunit II, transmembrane...",domain,16,111,1
849617,P93285,IPR014222,"Cytochrome c oxidase, subunit II",domain,28,249,1
849618,P93285,IPR034210,"Cytochrome c oxidase subunit 2, C-terminal",domain,114,248,1
849619,P93285,IPR036257,"Cytochrome C oxidase subunit II, transmembrane...",homologous_superfamily,2,111,1


In [ ]:
set(interpro['type'])

{'active_site',
 'binding_site',
 'conserved_site',
 'domain',
 'family',
 'homologous_superfamily',
 'ptm',
 'repeat'}

In [ ]:
from collections import defaultdict

# Define your custom type priority
type_order = {
    'homologous_superfamily': 0,
    'family': 1,
    'domain': 2,
    'conserved_site': 3,
    'ptm': 4,
    'repeat': 5,
    'active_site': 6,
    'binding_site': 7
}

# Intermediate mapping
accession_to_entries = defaultdict(list)

# Store everything per accession
for acc, ipr, start, end, typ in zip(
    interpro["accession"], interpro["interpro_id"],
    interpro["start"], interpro["end"], interpro["type"]
):
    accession_to_entries[acc].append((ipr, start, end, typ))

# Final output structure
collapsed_data = []

for acc, entries in accession_to_entries.items():
    interpro_ids = set()
    interpro_location = {}
    
    # Sort by (type priority, start)
    entries_sorted = sorted(
        entries,
        key=lambda x: (type_order.get(x[3], 99), x[1])  # (type, start)
    )

    for ipr, start, end, typ in entries_sorted:
        if ipr in interpro_location:
            raise ValueError(f"InterPro ID {ipr} appears more than once for protein {acc}")
        interpro_ids.add(ipr)
        interpro_location[ipr] = (start, end)

    collapsed_data.append({
        "accession": acc,
        "interpro_ids": list(interpro_ids),
        "interpro_location": interpro_location
    })

# Convert to DataFrame
collapsed_df = pd.DataFrame(collapsed_data)

In [ ]:
collapsed_df

,accession,interpro_ids,interpro_location
0,A0A8I5Y4S4,"[IPR011993, IPR038023, IPR000697, IPR014885]","{'IPR011993': (1, 117), 'IPR038023': (765, 803..."
1,Q67UU0,"[IPR003607, IPR043519, IPR007685, IPR006674]","{'IPR043519': (374, 563), 'IPR003607': (229, 3..."
2,Q08BW5,"[IPR036860, IPR001496, IPR000980, IPR036036, I...","{'IPR036860': (34, 171), 'IPR036036': (168, 21..."
3,Q53P49,"[IPR004993, IPR055377, IPR055378]","{'IPR004993': (14, 609), 'IPR055377': (358, 45..."
4,O95822,"[IPR038917, IPR007956, IPR038351, IPR042303, I...","{'IPR038351': (33, 191), 'IPR042303': (192, 49..."
...,...,...,...
190938,P34748,"[IPR019520, IPR023611]","{'IPR019520': (6, 133), 'IPR023611': (6, 129)}"
190939,P83088,"[IPR038577, IPR055270, IPR001503, IPR031481]","{'IPR038577': (166, 388), 'IPR001503': (73, 42..."
190940,Q9PW71,"[IPR016130, IPR020422, IPR000387, IPR029021, I...","{'IPR036873': (5, 146), 'IPR029021': (167, 321..."
190941,Q5R9X6,"[IPR000304, IPR028939, IPR053790, IPR008927, I...","{'IPR036291': (1, 162), 'IPR008927': (163, 270..."


## Update Train and Test with Interpro Data

In [ ]:
interpro_collapsed = collapsed_df.rename(columns={"accession": "protein_id"})

cafa5_train = cafa5_train.drop(columns=['interpro_ids'])
cafa5_train = cafa5_train.merge(interpro_collapsed, on="protein_id", how="left")

In [ ]:
cafa5_test = cafa5_test.drop(columns=['interpro_ids'])
cafa5_test = cafa5_test.merge(interpro_collapsed, on="protein_id", how="left")

In [ ]:
cafa5_test_super = cafa5_test_super.drop(columns=['interpro_ids'])
cafa5_test_super = cafa5_test_super.merge(interpro_collapsed, on="protein_id", how="left")

In [ ]:
print("Rows where train interpro is None:", cafa5_train["interpro_ids"].isnull().sum())
print("Rows where test interpro is None:", cafa5_test["interpro_ids"].isnull().sum())
print("Rows where test superset interpro is None:", cafa5_test_super["interpro_ids"].isnull().sum())

Rows where train interpro is None: 6581
Rows where test interpro is None: 2
Rows where test superset interpro is None: 5671


## Interpro Metadata

In [ ]:
interpro_metadata = pd.read_csv('interpro_lookup.tsv', sep='\t')

In [ ]:
interpro_metadata

,interpro_id,entry_name,type
0,IPR000697,WH1/EVH1 domain,domain
1,IPR011993,PH-like domain superfamily,homologous_superfamily
2,IPR014885,VASP tetramerisation,domain
3,IPR038023,VASP tetramerisation domain superfamily,homologous_superfamily
4,IPR003607,HD/PDEase domain,domain
...,...,...,...
36888,IPR039966,Uncharacterized protein C553.12c,family
36889,IPR010710,Protein of unknown function DUF1289,family
36890,IPR006475,"Citrate lyase, beta subunit, bacteria",family
36891,IPR004656,Hydroxymethylglutaryl-CoA synthase,family


## Upload Data

In [ ]:
#Making it into a json object so that huggingface can handle it
cafa5_train["interpro_location"] = cafa5_train["interpro_location"].apply(json.dumps)
cafa5_test["interpro_location"] = cafa5_test["interpro_location"].apply(json.dumps)
cafa5_test_super["interpro_location"] = cafa5_test_super["interpro_location"].apply(json.dumps)

In [ ]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)
test_super_hf = Dataset.from_pandas(cafa5_test_super, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf,
    "test_superset": test_super_hf
})

In [ ]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Upload CAFA5 with metadata. Uploaded InterPro domains with all metadata, including start and end and type"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/132M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/107M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/d3c5282669bc6ccb70f75e7be45c2d0cc5edb64a', commit_message='Upload CAFA5 with metadata v1.3. Uploaded InterPro domains and Structure paths', commit_description='', oid='d3c5282669bc6ccb70f75e7be45c2d0cc5edb64a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
interpro_hf = Dataset.from_pandas(interpro_metadata, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "metadata": interpro_hf,
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interpro_metadata",
    commit_message="Upload the InterPro Metadata, downloaded from the Uniprot Cross references"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/755k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/854b6f68c252aaab2eeb5696a504827718077278', commit_message='Upload the InterPro Metadata, downloaded from the Uniprot Cross references', commit_description='', oid='854b6f68c252aaab2eeb5696a504827718077278', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Adding in Interpro Descriptions

In [ ]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from tqdm import tqdm
import re
from bs4 import BeautifulSoup


def clean_description(raw_desc):
    """Remove HTML tags and citations from InterPro descriptions."""
    if not raw_desc:
        return None

    # Remove HTML tags (<p>, etc.)
    text = BeautifulSoup(raw_desc, "html.parser").get_text(separator="\n")

    # Remove [cite:XXXX] patterns
    text = re.sub(r"\[\[cite:[^\]]+\]\]", "", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def fetch_interpro_description(interpro_id, max_retries=3):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/InterPro/{interpro_id}"
    headers = {"Accept": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 204:
                return {"interpro_id": interpro_id, "description": None}

            r.raise_for_status()
            data = r.json()

            description_raw = data.get("metadata", {}).get("description")
            description = clean_description(description_raw)
            name = data.get("metadata", {}).get("name")
            entry_type = data.get("metadata", {}).get("type")

            return {
                "interpro_id": interpro_id,
                "name": name,
                "type": entry_type,
                "description": description
            }

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Failed for {interpro_id}: {e}")
            sleep(2 * (attempt + 1))

    return {"interpro_id": interpro_id, "description": None}


def fetch_all_descriptions(interpro_ids, output_file="interpro_descriptions.tsv", max_threads=16):
    results = []

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(fetch_interpro_description, ipr): ipr for ipr in interpro_ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching InterPro Descriptions"):
            results.append(future.result())

    df = pd.DataFrame(results)
    df.to_csv(output_file, sep="\t", index=False)
    print(f"✅ Done. Fetched {df['description'].notna().sum()} clean descriptions out of {len(interpro_ids)} IDs")
    return df


if __name__ == "__main__":
    ds = load_dataset("wanglab/cafa5", "interpro_metadata")
    interpro = ds["metadata"].to_pandas()

    intepro_ids = set(list(interpro["interpro_id"]))

    print(f"Fetched {len(intepro_ids)} intepro ids")

    fetch_all_interpro_parallel(
        list(intepro_ids),
        output_file="interpro_descriptions.tsv",
        interpro_lookup_file="interpro_lookup_descriptions.tsv",
        max_threads=32
    )

## Interpro Metadata

In [2]:
import pandas as pd

In [4]:
interpro_metadata = pd.read_csv('interpro_descriptions.tsv', sep='\t')

In [5]:
interpro_metadata

,interpro_id,name,type,description
0,IPR051901,"{'name': 'Protease Inhibitor I33', 'short': 'P...",family,{'text': ' The protease inhibitor I33 family c...
1,IPR045108,{'name': 'Thioredoxin domain-containing protei...,family,{'text': ' This entry represents a group of th...
2,IPR038492,"{'name': 'GBBH-like, N-terminal domain superfa...",homologous_superfamily,{'text': ' Gamma-butyrobetaine hydroxylase (GB...
3,IPR045527,{'name': 'Protein of unknown function DUF6470'...,family,{'text': ' This family of proteins is function...
4,IPR020608,"{'name': 'RNA polymerase, subunit H/Rpb5, cons...",conserved_site,{'text': ' Prokaryotes contain a single DNA-de...
...,...,...,...,...
36888,IPR013483,{'name': 'Molybdenum cofactor biosynthesis pro...,family,"{'text': "" This entry represents the MoaA prot..."
36889,IPR052641,{'name': 'A-kinase anchor protein 7 isoform ga...,family,{'text': ' This family of proteins is involved...
36890,IPR004183,"{'name': 'Extradiol ring-cleavage dioxygenase,...",domain,"{'text': "" Dioxygenases catalyse the incorpora..."
36891,IPR006520,"{'name': 'Distal tail protein, N-terminal', 's...",domain,{'text': ' This entry represents the N-termina...


In [10]:
import ast
import json
import re
import pandas as pd

def to_dict_safe(x):
    """
    Convert a value that may be:
      - already a dict
      - a JSON string (double quotes)
      - a Python-literal string (single quotes)
      - something else (try regex extracting 'name' or 'text')
    Returns a dict (possibly empty) rather than None.
    """
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    s = str(x).strip()

    # try JSON first (valid JSON uses double quotes)
    try:
        return json.loads(s)
    except Exception:
        pass

    # try ast.literal_eval for Python-literal strings like "{'name': 'Foo'}"
    try:
        return ast.literal_eval(s)
    except Exception:
        pass

    # fallback: try to extract common fields with regex (single/double quotes)
    d = {}
    m = re.search(r"""['"]name['"]\s*:\s*['"]([^'"]+)['"]""", s)
    if m:
        d['name'] = m.group(1).strip()
    m2 = re.search(r"""['"]text['"]\s*:\s*['"]([^'"]+)['"]""", s)
    if m2:
        d['text'] = m2.group(1).strip()
    return d

# Apply conversion to your DataFrame
interpro_metadata['name_dict'] = interpro_metadata['name'].apply(to_dict_safe)
interpro_metadata['description_dict'] = interpro_metadata['description'].apply(to_dict_safe)

# Extract fields
interpro_metadata['entry_name'] = interpro_metadata['name_dict'].apply(lambda d: d.get('name'))
interpro_metadata['description_text'] = interpro_metadata['description_dict'].apply(lambda d: d.get('text'))

# Optional: strip whitespace and normalize newlines in description_text
interpro_metadata['description_text'] = interpro_metadata['description_text'].astype(object).apply(
    lambda t: t.strip() if isinstance(t, str) else t
)

# (Optional) drop helper dict columns if you don't need them
interpro_metadata = interpro_metadata.drop(columns=['name_dict', 'description_dict'])

In [11]:
interpro_metadata

,interpro_id,name,type,description,entry_name,description_text
0,IPR051901,"{'name': 'Protease Inhibitor I33', 'short': 'P...",family,{'text': ' The protease inhibitor I33 family c...,Protease Inhibitor I33,The protease inhibitor I33 family comprises pr...
1,IPR045108,{'name': 'Thioredoxin domain-containing protei...,family,{'text': ' This entry represents a group of th...,Thioredoxin domain-containing protein 17-like,This entry represents a group of thioredoxin d...
2,IPR038492,"{'name': 'GBBH-like, N-terminal domain superfa...",homologous_superfamily,{'text': ' Gamma-butyrobetaine hydroxylase (GB...,"GBBH-like, N-terminal domain superfamily","Gamma-butyrobetaine hydroxylase (GBBH), also k..."
3,IPR045527,{'name': 'Protein of unknown function DUF6470'...,family,{'text': ' This family of proteins is function...,Protein of unknown function DUF6470,This family of proteins is functionally unchar...
4,IPR020608,"{'name': 'RNA polymerase, subunit H/Rpb5, cons...",conserved_site,{'text': ' Prokaryotes contain a single DNA-de...,"RNA polymerase, subunit H/Rpb5, conserved site",Prokaryotes contain a single DNA-dependent RNA...
...,...,...,...,...,...,...
36888,IPR013483,{'name': 'Molybdenum cofactor biosynthesis pro...,family,"{'text': "" This entry represents the MoaA prot...",Molybdenum cofactor biosynthesis protein A,This entry represents the MoaA protein (molybd...
36889,IPR052641,{'name': 'A-kinase anchor protein 7 isoform ga...,family,{'text': ' This family of proteins is involved...,A-kinase anchor protein 7 isoform gamma,This family of proteins is involved in targeti...
36890,IPR004183,"{'name': 'Extradiol ring-cleavage dioxygenase,...",domain,"{'text': "" Dioxygenases catalyse the incorpora...","Extradiol ring-cleavage dioxygenase, class III...",Dioxygenases catalyse the incorporation of bot...
36891,IPR006520,"{'name': 'Distal tail protein, N-terminal', 's...",domain,{'text': ' This entry represents the N-termina...,"Distal tail protein, N-terminal",This entry represents the N-terminal domain of...


In [13]:
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "interpro_metadata")
interpro = ds["metadata"].to_pandas()

README.md: 0.00B [00:00, ?B/s]

interpro_metadata/metadata-00000-of-0000(…):   0%|          | 0.00/1.07M [00:00<?, ?B/s]

Generating metadata split:   0%|          | 0/36893 [00:00<?, ? examples/s]

In [34]:
merged = pd.merge(interpro, interpro_metadata, on="interpro_id", how="left")

In [35]:
merged

,interpro_id,entry_name_x,type_x,name,type_y,description,entry_name_y,description_text
0,IPR000697,WH1/EVH1 domain,domain,"{'name': 'WH1/EVH1 domain', 'short': 'WH1/EVH1...",domain,"{'text': ' The EVH1 (WH1, RanBP1-WASP) domain ...",WH1/EVH1 domain,"The EVH1 (WH1, RanBP1-WASP) domain is found in..."
1,IPR011993,PH-like domain superfamily,homologous_superfamily,"{'name': 'PH-like domain superfamily', 'short'...",homologous_superfamily,{'text': ' This superfamily represents the PH ...,PH-like domain superfamily,This superfamily represents the PH domain and ...
2,IPR014885,VASP tetramerisation,domain,"{'name': 'VASP tetramerisation', 'short': 'VAS...",domain,{'text': ' Vasodilator-stimulated phosphoprote...,VASP tetramerisation,Vasodilator-stimulated phosphoprotein (VASP) i...
3,IPR038023,VASP tetramerisation domain superfamily,homologous_superfamily,{'name': 'VASP tetramerisation domain superfam...,homologous_superfamily,{'text': ' Vasodilator-stimulated phosphoprote...,VASP tetramerisation domain superfamily,Vasodilator-stimulated phosphoprotein (VASP) i...
4,IPR003607,HD/PDEase domain,domain,"{'name': 'HD/PDEase domain', 'short': 'HD/PDEa...",domain,"{'text': "" This entry represents the HD domain...",HD/PDEase domain,"This entry represents the HD domain, which is ..."
...,...,...,...,...,...,...,...,...
36888,IPR039966,Uncharacterized protein C553.12c,family,"{'name': 'Uncharacterized protein C553.12c', '...",family,{'text': ' The function of this family of fung...,Uncharacterized protein C553.12c,The function of this family of fungal proteins...
36889,IPR010710,Protein of unknown function DUF1289,family,{'name': 'Protein of unknown function DUF1289'...,family,{'text': ' This family consists of a number of...,Protein of unknown function DUF1289,This family consists of a number of hypothetic...
36890,IPR006475,"Citrate lyase, beta subunit, bacteria",family,"{'name': 'Citrate lyase, beta subunit, bacteri...",family,{'text': ' This group of sequences represent t...,"Citrate lyase, beta subunit, bacteria",This group of sequences represent the beta sub...
36891,IPR004656,Hydroxymethylglutaryl-CoA synthase,family,"{'name': 'Hydroxymethylglutaryl-CoA synthase',...",family,{'text': ' This entry represents Hydroxymethyl...,Hydroxymethylglutaryl-CoA synthase,This entry represents Hydroxymethylglutaryl-Co...


In [38]:
merged = merged.drop(columns=['type_y','description', 'entry_name_y', 'name'])

In [39]:
merged = merged.rename(columns={'entry_name_x': 'entry_name', 'type_x': 'type', })

In [40]:
merged

,interpro_id,entry_name,type,description_text
0,IPR000697,WH1/EVH1 domain,domain,"The EVH1 (WH1, RanBP1-WASP) domain is found in..."
1,IPR011993,PH-like domain superfamily,homologous_superfamily,This superfamily represents the PH domain and ...
2,IPR014885,VASP tetramerisation,domain,Vasodilator-stimulated phosphoprotein (VASP) i...
3,IPR038023,VASP tetramerisation domain superfamily,homologous_superfamily,Vasodilator-stimulated phosphoprotein (VASP) i...
4,IPR003607,HD/PDEase domain,domain,"This entry represents the HD domain, which is ..."
...,...,...,...,...
36888,IPR039966,Uncharacterized protein C553.12c,family,The function of this family of fungal proteins...
36889,IPR010710,Protein of unknown function DUF1289,family,This family consists of a number of hypothetic...
36890,IPR006475,"Citrate lyase, beta subunit, bacteria",family,This group of sequences represent the beta sub...
36891,IPR004656,Hydroxymethylglutaryl-CoA synthase,family,This entry represents Hydroxymethylglutaryl-Co...


## Upload Data

In [41]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
interpro_hf = Dataset.from_pandas(interpro_metadata, preserve_index=False)

In [42]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "metadata": interpro_hf,
})

In [43]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interpro_metadata",
    commit_message="Upload the InterPro Metadata for the latest function summaries."
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/8b86a64b949654c8f5cfe00eb8d90d6a0b2539a7', commit_message='Upload the InterPro Metadata for the latest function summaries.', commit_description='', oid='8b86a64b949654c8f5cfe00eb8d90d6a0b2539a7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Fixing Structure Path

In [ ]:
import pandas as pd

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

README.md: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/133496 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa5_train = ds['train'].to_pandas()
cafa5_test = ds['test'].to_pandas()

In [ ]:
cafa5_train['structure_path'] = cafa5_train['structure_path'].apply(
    lambda x: x.split('/')[-1].split('.gz')[0] if pd.notna(x) else x
)
cafa5_test['structure_path'] = cafa5_test['structure_path'].apply(
    lambda x: x.split('/')[-1].split('.gz')[0] if pd.notna(x) else x
)

In [ ]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [ ]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Fixed structure paths"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/131M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/107M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/4875a84b5abde0068f489d0389bedca940d043c1', commit_message='Fixed structure paths', commit_description='', oid='4875a84b5abde0068f489d0389bedca940d043c1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Download AlphaFold Structures for all Proteins

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

README.md: 0.00B [00:00, ?B/s]

cafa5_reasoning/train-00000-of-00001.par(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

cafa5_reasoning/test-00000-of-00001.parq(…):   0%|          | 0.00/102M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133496 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [ ]:
protein_ids = set(list(ds['train']['protein_id']) + list(ds['test']['protein_id']))
len(protein_ids)

201777

In [ ]:
print(f"Number of proteins that overlap between test and train set: {len(list(set(list(ds['train']['protein_id'])) & set(list(set(list(ds['train']['protein_id']))))))}")

Number of proteins that overlap between test and train set: 133496


### Downloading AlphaFoldDB

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [ ]:
with open("alphafold_cif_manifest.txt", "w") as f:
    for protein in (protein_ids):
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{protein}-F1-model_v4.cif\n")

In [ ]:
cat alphafold_cif_manifest.txt | gsutil -m cp -I af_db_full/

In [ ]:
import os

# Combined list of all expected accessions
all_expected = protein_ids

# Get all filenames in af_db/ and extract accessions
downloaded_files = os.listdir("af_db")
downloaded_accessions = set(
    filename.split("-")[1]  # Extract accession from 'AF-<ACCESSION>-F1-model_v4.cif'
    for filename in downloaded_files
    if filename.endswith(".cif")
)

# Find accessions that are missing
missing_accessions = sorted(all_expected - downloaded_accessions)

print(f"Expected: {len(all_expected)}")
print(f"Downloaded: {len(downloaded_accessions)}")
print(f"Missing: {len(missing_accessions)}")

# Save to file
with open("missing_af_accessions.txt", "w") as f:
    for acc in missing_accessions:
        f.write(f"{acc}\n")

Expected: 201777
Downloaded: 193149
Missing: 8628


In [ ]:
mkdir -p af_shards

i=0
shard=0

for file in af_db_full/*.cif; do
    # Create a new folder every 5000 files
    if (( i % 5000 == 0 )); then
        shard_dir="af_shards/shard_${shard}"
        mkdir -p "$shard_dir"
        ((shard++))
    fi

    cp "$file" "$shard_dir/"
    ((i++))
done

In [ ]:
cd af_shards

ls -d shard_* | parallel --bar 'tar -czf {}.tar.gz {} && rm -r {}'

cd ..

In [ ]:
cd ../../
git lfs install
git clone https://huggingface.co/datasets/wanglab/cafa5

mkdir cafa5/structures
cp -r af_shards/ ../../cafa5/structures_af/

cd ../../cafa5/

git add structures_af/
git commit -m "Add AlphaFold structures for most proteins"
git push

In [ ]:
import pandas as pd

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa5_train = ds['train'].to_pandas()
cafa5_test = ds['test'].to_pandas()

In [ ]:
len(missing_accessions)

8628

In [ ]:
cafa5_train['structure_path'] = cafa5_train['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v4.cif" if x not in missing_accessions else None
)
cafa5_test['structure_path'] = cafa5_test['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v4.cif" if x not in missing_accessions else None
)

In [ ]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [ ]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Fixed structure paths to only AF paths"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/127M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/102M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/9e3403f7b2462baadbb4803ebd1186e28a9698ce', commit_message='Fixed structure paths to only AF paths', commit_description='', oid='9e3403f7b2462baadbb4803ebd1186e28a9698ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Add in structures for those with mismatches structures or absent ones

Ran load.py to download all structures from structures_af

Script to check if structures are present and if the lengths are the same for the protein sequence and the structure

In [ ]:
import pandas as pd
import numpy as np
import os
from Bio.PDB import MMCIFParser, is_aa
from datasets import load_dataset
import concurrent.futures
from datetime import datetime

def _coords_from_cif(cif_path, chain_id="A", atom_order=["N", "CA", "C"]):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("protein", cif_path)
    model = structure[0]
    if chain_id not in model:
        raise ValueError(f"Chain {chain_id} not found in {cif_path}")
    chain = model[chain_id]
    coords = []
    for residue in chain:
        if not is_aa(residue, standard=True):
            continue
        atom_coords = []
        for atom_name in atom_order:
            try:
                coord = residue[atom_name].get_coord()
            except KeyError:
                coord = np.full(3, np.nan)
            atom_coords.append(coord)
        coords.append(atom_coords)
    coords = np.array(coords)
    return coords

def check_structure_row(row, structure_dir=None, chain_id="A", atom_order=["N", "CA", "C"]):
    protein_id = row.get("protein_id")
    sequence = row.get("sequence")
    structure_path = row.get("structure_path")
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Processing {protein_id}", flush=True)
    
    if pd.isna(sequence) or pd.isna(structure_path):
        print(f"[{protein_id}] ⚠️  Skipped: missing sequence or structure_path", flush=True)
        return {"protein_id": protein_id, "structure_correct": False}
    
    cif_path = structure_path
    if structure_dir is not None and not os.path.isabs(structure_path):
        cif_path = os.path.join(structure_dir, structure_path)
    
    print(f"[{protein_id}] 📁 Reading CIF: {cif_path}", flush=True)
    
    try:
        coords = _coords_from_cif(cif_path, chain_id=chain_id, atom_order=atom_order)
        structure_len = coords.shape[0]
        seq_len = len(sequence)
        structure_correct = (seq_len == structure_len)
        
        if structure_correct:
            print(f"[{protein_id}] ✅ Success: sequence={seq_len}, structure={structure_len}", flush=True)
        else:
            print(f"[{protein_id}] ❌ Length mismatch: sequence={seq_len}, structure={structure_len}", flush=True)
            
    except Exception as e:
        print(f"[{protein_id}] 💥 Failed to parse structure: {str(e)}", flush=True)
        structure_correct = False

    return {"protein_id": protein_id, "structure_correct": structure_correct}

def check_structure_correctness_parallel(df, structure_dir=None, chain_id="A", atom_order=["N", "CA", "C"], n_jobs=8):
    print(f"🚀 Starting parallel processing with {n_jobs} workers", flush=True)
    print(f"📊 Total rows to process: {len(df)}", flush=True)
    
    filtered_df = df.dropna(subset=["sequence", "structure_path"])
    print(f"🔍 Filtered rows with valid data: {len(filtered_df)}", flush=True)
    
    results = []
    start_time = datetime.now()
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        print(f"⚡ Submitting {len(filtered_df)} jobs to process pool...", flush=True)
        
        futures = [
            executor.submit(check_structure_row, row, structure_dir, chain_id, atom_order)
            for _, row in filtered_df.iterrows()
        ]
        
        print("📋 All jobs submitted, waiting for completion...", flush=True)
        
        completed = 0
        for f in concurrent.futures.as_completed(futures):
            try:
                result = f.result()
                results.append(result)
                completed += 1
                
                if completed % 100 == 0:
                    elapsed = datetime.now() - start_time
                    rate = completed / elapsed.total_seconds()
                    eta_seconds = (len(filtered_df) - completed) / rate if rate > 0 else 0
                    eta = datetime.now().timestamp() + eta_seconds
                    eta_str = datetime.fromtimestamp(eta).strftime('%H:%M:%S')
                    
                    print(f"📈 Progress: {completed}/{len(filtered_df)} ({completed/len(filtered_df)*100:.1f}%) "
                          f"- Rate: {rate:.1f} proteins/sec - ETA: {eta_str}", flush=True)
                    
            except Exception as e:
                print(f"💥 Error in worker: {str(e)}", flush=True)
                # Add a placeholder result to maintain count
                results.append({"protein_id": "ERROR", "structure_correct": False})

    print(f"✅ Parallel processing completed in {datetime.now() - start_time}", flush=True)
    
    # Add back rows that were dropped due to NaNs, marked as incorrect
    missing_rows = df[df[["sequence", "structure_path"]].isna().any(axis=1)]
    print(f"⚠️  Adding back {len(missing_rows)} rows with missing data", flush=True)
    
    for _, row in missing_rows.iterrows():
        results.append({"protein_id": row.get("protein_id"), "structure_correct": False})
    
    print(f"📊 Final result count: {len(results)}", flush=True)
    return pd.DataFrame(results)

# ---------- Main execution ----------
if __name__ == "__main__":
    print("=" * 80)
    print("🧬 CAFA5 Structure Validation Script")
    print("=" * 80)
    
    print("📥 Loading datasets...", flush=True)
    ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")
    cafa5_train = ds["train"].to_pandas()
    cafa5_test = ds["test"].to_pandas()
    cafa5_test_super = ds["test_superset"].to_pandas()

    all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
    all = all.drop(columns=['protein_names','protein_function','organism','length','subcellular_location','go_ids','go_bp','go_mf','go_cc','string_id','interaction_partners','full_interaction_info','interpro_ids','interpro_location'])
    all.drop_duplicates(inplace=True)
    all.reset_index(drop=True, inplace=True)

    print(f"Rows in all: {len(all)}")

    # Optionally set the directory where CIF files are stored
    structure_dir = "data/extracted/"
    n_jobs = 32
    
    print("🔧 Configuration:")
    print(f"   Structure directory: {structure_dir}")
    print(f"   Number of workers: {n_jobs}")
    print("   Chain ID: A")
    print("   Atom order: ['N', 'CA', 'C']")

    print("\n" + "=" * 80)
    print("🚀 Starting structure validation...")
    print("=" * 80)

    print("\n📋 Processing all set...")
    test_structure_train = check_structure_correctness_parallel(all, structure_dir, n_jobs=n_jobs)
    
    # Save train results immediately
    print("💾 Saving Final results...")
    test_structure_train.to_csv("validated_structures_all.csv", index=False)
    
    # Analyze and save train failed proteins immediately
    failed_train = set(test_structure_train.loc[~test_structure_train["structure_correct"], "protein_id"])
    print(f"❌ Failed proteins: {len(failed_train)}")
    with open("failed_structures_all.txt", "w") as f:
        for pid in sorted(failed_train):
            f.write(f"{pid}\n")
    print("✅ Failed proteins saved to failed_structures_all.txt")
    
    print("\n📊 Final analysis...")
    print("❌ Failed proteins summary:")
    print(f"   All: {len(failed_train)}")

    print("\n" + "=" * 80)
    print("🎉 Structure validation completed!")
    print("=" * 80)

Now getting the seqeunces for the ones that failed my tests. 

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [2]:
ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

README.md: 0.00B [00:00, ?B/s]

cafa5_reasoning/train-00000-of-00002.par(…):   0%|          | 0.00/105M [00:00<?, ?B/s]

cafa5_reasoning/train-00001-of-00002.par(…):   0%|          | 0.00/78.3M [00:00<?, ?B/s]

cafa5_reasoning/test-00000-of-00001.parq(…):   0%|          | 0.00/396k [00:00<?, ?B/s]

cafa5_reasoning/test_superset-00000-of-0(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133538 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/264 [00:00<?, ? examples/s]

Generating test_superset split:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [3]:
cafa5_train = ds["train"].to_pandas()
cafa5_test = ds["test"].to_pandas()
cafa5_test_super = ds["test_superset"].to_pandas()

all = pd.concat([cafa5_train, cafa5_test, cafa5_test_super])
all = all.drop(columns=['protein_names','protein_function','organism','length','subcellular_location','go_ids','go_bp','go_mf','go_cc','string_id','interaction_partners','full_interaction_info','interpro_ids','interpro_location'])
all.drop_duplicates(inplace=True)
all.reset_index(drop=True, inplace=True)

In [4]:
all

,protein_id,sequence,structure_path
0,A0A087X1C5,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,AF-A0A087X1C5-F1-model_v4.cif
1,A0A0B4J2F0,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,AF-A0A0B4J2F0-F1-model_v4.cif
2,A0A0A7EPL0,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,AF-A0A0A7EPL0-F1-model_v4.cif
3,A0A0B4J1G0,MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,AF-A0A0B4J1G0-F1-model_v4.cif
4,A0A0B4J1N3,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,AF-A0A0B4J1N3-F1-model_v4.cif
...,...,...,...
201772,Q4A3R3,MGTSAVILEICLLLSQVLTTVSSTTQTESTTEDRTQITETAFWETQ...,AF-Q4A3R3-F1-model_v4.cif
201773,Q5E9L2,MNWRFVELLYFLFVWGRISVQPSHQEPAATDQHVSKEFDWLISDRG...,AF-Q5E9L2-F1-model_v4.cif
201774,Q3T0K9,MNSKGQYPTQPTYPVQPPGNPVYPQTLHLPQAPPYTDAPPAYSELY...,AF-Q3T0K9-F1-model_v4.cif
201775,Q58CX7,MEPRLGPKAAALHLGWPFLLLWVSGLSYSVSSPASPSPSPVSRVRT...,AF-Q58CX7-F1-model_v4.cif


In [5]:
with open("failed_structures_all.txt", "r") as f:
    protein_list = [line.strip() for line in f]

In [6]:
len(protein_list)

10030

In [7]:
# Filter the DataFrame for only those proteins
filtered = all[all['protein_id'].isin(protein_list)]

# Create a dictionary: {protein_id: sequence}
protein_to_sequence = dict(zip(filtered['protein_id'], filtered['sequence']))

In [8]:
len(protein_to_sequence)

10030

Save the fasta files to dir

In [9]:
import os

# Directory to save FASTA files
output_dir = "<output_dir>/fasta_files"
os.makedirs(output_dir, exist_ok=True)

# Write each protein to a separate FASTA file
for protein_id, sequence in protein_to_sequence.items():
    fasta_path = os.path.join(output_dir, f"{protein_id}.fasta")
    with open(fasta_path, "w") as f:
        f.write(f">A|protein\n")
        f.write(f"{sequence}\n")

# Creating Local MSAs for all 10k proteins without structures

ColabFold DB (colabfold_envdb_202108)

In [ ]:
aws s3 cp --no-sign-request s3://steineggerlab/colabfold/colabfold_envdb_2023.tar.gz .
tar -xvzf colabfold_envdb_2023.tar.gz

In [ ]:
mkdir -p databases tmp msa_output logs

In [ ]:
#!/bin/bash

INPUT_DIR=fasta_files
OUTPUT_DIR=msa_output
TMP_DIR=tmp
DB_DIR=databases
DB_PATH=colabfold_envdb_2023/colabfold_envdb_2023

mkdir -p "$OUTPUT_DIR" "$TMP_DIR" "$DB_DIR"

process_one() {
    fasta_file="$1"
    base_name=$(basename "$fasta_file" .fasta)

    mmseqs createdb "$fasta_file" "$DB_DIR/$base_name" "$TMP_DIR/$base_name"

    mmseqs search "$DB_DIR/$base_name" "$DB_PATH" "$DB_DIR/${base_name}_result" "$TMP_DIR/$base_name" \
        --threads 1 -s 7.5 --max-seqs 8000

    mmseqs result2msa "$DB_DIR/$base_name" "$DB_PATH" "$DB_DIR/${base_name}_result" \
        "$OUTPUT_DIR/$base_name.a3m" "$TMP_DIR/$base_name" \
        --msa-format-mode 0
}

export -f process_one
export DB_DIR TMP_DIR OUTPUT_DIR DB_PATH

find "$INPUT_DIR" -name '*.fasta' | parallel -j 16 process_one {}

Boltz2 command to run

In [ ]:
boltz predict data/structures_failed/fasta_files \
    --out_dir data/structures_failed/ \
    --cache ~/.cache/boltz2 --devices 2 --use_msa_server